In [9]:
import os
import json
from pathlib import Path
import pandas as pd

In [10]:
DATA_DIR = Path("/Users/jekaterinasergejeva/Desktop/Masters/SHROOM_CAP/data")

FILES = [
    "en_test_data.jsonl",
    "en_train_data.jsonl",
    "en_train_label.jsonl",
    "en_valid_label.jsonl",
]

In [11]:
def _load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid JSON on line {i} in {path.name}: {e}") from e
    return pd.DataFrame(rows)

def explore_file(filename: str) -> dict:
    path = DATA_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    df = _load_jsonl(path)

    info = {
        "file": filename,
        "path": str(path),
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "column_names": list(df.columns),
        "summary": _summarize_df(df),
        "head": df.head(5),
    }
    return info

def _safe_unique_count(s: pd.Series) -> int:
    """
    Count unique values even if the series contains unhashable types (list/dict).
    """
    try:
        return int(s.nunique(dropna=True))
    except TypeError:
        def _to_hashable(x):
            if pd.isna(x):
                return None
            if isinstance(x, (list, dict)):
                # stable representation for nested JSON
                return json.dumps(x, sort_keys=True, ensure_ascii=False)
            return x

        return int(s.map(_to_hashable).nunique(dropna=True))

def _summarize_df(df: pd.DataFrame) -> pd.DataFrame:
    missing = df.isna().sum()
    non_null = df.notna().sum()

    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing_values": missing,
        "missing_pct": (missing / len(df) * 100).round(2) if len(df) else 0.0,
        "non_null": non_null,
        "unique": [ _safe_unique_count(df[c]) for c in df.columns ],
    }, index=df.columns).sort_index()

    return summary


In [12]:
results = []
for fn in FILES:
    res = explore_file(fn)
    results.append(res)

# Pretty print results per file
for res in results:
    print("=" * 100)
    print(f"FILE: {res['file']}")
    print(f"PATH: {res['path']}")
    print(f"SHAPE: {res['rows']} rows x {res['columns']} columns")
    print(f"COLUMNS: {res['column_names']}")
    print("-" * 100)
    display(res["summary"])
    print("-" * 100)
    display(res["head"])

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()